# Option Pricing - Tests et convergences

Ce notebook sert de banc de test interactif pour les pricers du repo :

- Black-Scholes-Merton pour options europeennes.
- Arbre binomial CRR pour options europeennes et americaines.
- Longstaff-Schwartz Monte Carlo pour options americaines.

Depuis Anaconda Prompt, lance d'abord :

```powershell
cd "C:\\Users\\hmaur\\Documents\\Option Pricing"
conda activate option-pricing
python -m pip install -e ".[dev]"
jupyter notebook
```

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if (repo_root / "src").exists():
    sys.path.insert(0, str(repo_root / "src"))

import matplotlib.pyplot as plt
import numpy as np

from option_pricing import (
    ExerciseStyle,
    MarketInputs,
    OptionContract,
    OptionType,
    binomial_price,
    black_scholes_price,
    lsmc_price,
)

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Parametres de base

In [ ]:
market = MarketInputs(rate=0.05, volatility=0.20, dividend_yield=0.02)

european_call = OptionContract(
    spot=100,
    strike=100,
    maturity=1.0,
    option_type=OptionType.CALL,
    exercise_style=ExerciseStyle.EUROPEAN,
)

european_put = OptionContract(
    spot=100,
    strike=100,
    maturity=1.0,
    option_type=OptionType.PUT,
    exercise_style=ExerciseStyle.EUROPEAN,
)

american_call = OptionContract(
    spot=100,
    strike=100,
    maturity=1.0,
    option_type=OptionType.CALL,
    exercise_style=ExerciseStyle.AMERICAN,
)

american_put = OptionContract(
    spot=100,
    strike=100,
    maturity=1.0,
    option_type=OptionType.PUT,
    exercise_style=ExerciseStyle.AMERICAN,
)

market, european_call

## 2. Smoke tests numeriques

In [ ]:
bs_call = black_scholes_price(european_call, market)
bs_put = black_scholes_price(european_put, market)
tree_call = binomial_price(european_call, market, steps=750).price
tree_put = binomial_price(european_put, market, steps=750).price
american_put_tree = binomial_price(american_put, market, steps=750).price

print(f"Black-Scholes call europeen : {bs_call:.6f}")
print(f"Binomial call europeen      : {tree_call:.6f}")
print(f"Black-Scholes put europeen  : {bs_put:.6f}")
print(f"Binomial put europeen       : {tree_put:.6f}")
print(f"Binomial put americain      : {american_put_tree:.6f}")

assert abs(tree_call - bs_call) < 0.03
assert abs(tree_put - bs_put) < 0.03
assert american_put_tree >= tree_put

## 3. Parite put-call europeenne avec dividende continu

Pour options europeennes avec dividend yield continu :

$$ C - P = S e^{-qT} - K e^{-rT} $$

In [ ]:
lhs = bs_call - bs_put
rhs = european_call.spot * np.exp(-market.dividend_yield * european_call.maturity) - european_call.strike * np.exp(-market.rate * european_call.maturity)

print(f"C - P                        : {lhs:.10f}")
print(f"S exp(-qT) - K exp(-rT)      : {rhs:.10f}")
print(f"Erreur absolue               : {abs(lhs - rhs):.2e}")

assert abs(lhs - rhs) < 1e-10

## 4. Convergence binomiale vers Black-Scholes

Ici on price le meme call europeen avec un nombre croissant de pas binomiaux. Le prix doit converger vers Black-Scholes.

In [ ]:
steps_grid = np.array([5, 10, 25, 50, 100, 200, 500, 1000])
binomial_calls = np.array([binomial_price(european_call, market, steps=int(n)).price for n in steps_grid])
errors = np.abs(binomial_calls - bs_call)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(steps_grid, binomial_calls, marker="o", label="Binomial CRR")
axes[0].axhline(bs_call, color="black", linestyle="--", label="Black-Scholes")
axes[0].set_xlabel("Nombre de pas")
axes[0].set_ylabel("Prix")
axes[0].set_title("Prix binomial vs Black-Scholes")
axes[0].legend()

axes[1].loglog(steps_grid, errors, marker="o")
axes[1].set_xlabel("Nombre de pas")
axes[1].set_ylabel("Erreur absolue")
axes[1].set_title("Erreur de convergence")

plt.show()

list(zip(steps_grid, np.round(binomial_calls, 6), np.round(errors, 6)))

## 5. Prime d'exercice anticipe pour un put americain

Un put americain doit valoir au moins autant que le put europeen correspondant.

In [ ]:
strikes = np.arange(70, 131, 5)
euro_put_prices = []
american_put_prices = []

for strike in strikes:
    euro = OptionContract(100, strike, 1.0, OptionType.PUT, ExerciseStyle.EUROPEAN)
    amer = OptionContract(100, strike, 1.0, OptionType.PUT, ExerciseStyle.AMERICAN)
    euro_put_prices.append(binomial_price(euro, market, steps=500).price)
    american_put_prices.append(binomial_price(amer, market, steps=500).price)

euro_put_prices = np.array(euro_put_prices)
american_put_prices = np.array(american_put_prices)
early_exercise_premium = american_put_prices - euro_put_prices

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(strikes, euro_put_prices, marker="o", label="Put europeen")
axes[0].plot(strikes, american_put_prices, marker="o", label="Put americain")
axes[0].set_xlabel("Strike")
axes[0].set_ylabel("Prix")
axes[0].set_title("Prix put europeen vs americain")
axes[0].legend()

axes[1].bar(strikes, early_exercise_premium, width=3.5)
axes[1].set_xlabel("Strike")
axes[1].set_ylabel("Prime")
axes[1].set_title("Prime d'exercice anticipe")

plt.show()

assert np.all(early_exercise_premium >= -1e-10)

## 6. Call americain sans dividende vs call europeen

Sans dividendes, un call americain sur action a le meme prix que le call europeen : l'exercice anticipe n'est pas optimal.

In [ ]:
no_div_market = MarketInputs(rate=0.05, volatility=0.20, dividend_yield=0.0)
euro_no_div_call = OptionContract(100, 100, 1.0, OptionType.CALL, ExerciseStyle.EUROPEAN)
amer_no_div_call = OptionContract(100, 100, 1.0, OptionType.CALL, ExerciseStyle.AMERICAN)

bs_no_div_call = black_scholes_price(euro_no_div_call, no_div_market)
tree_amer_no_div_call = binomial_price(amer_no_div_call, no_div_market, steps=1000).price

print(f"Black-Scholes call europeen sans dividende : {bs_no_div_call:.6f}")
print(f"Binomial call americain sans dividende     : {tree_amer_no_div_call:.6f}")
print(f"Difference                                : {tree_amer_no_div_call - bs_no_div_call:.6f}")

assert abs(tree_amer_no_div_call - bs_no_div_call) < 0.02

## 7. Effet du dividend yield sur un call et un put europeens

In [ ]:
q_grid = np.linspace(0.0, 0.08, 17)
call_by_q = []
put_by_q = []

for q in q_grid:
    m = MarketInputs(rate=0.05, volatility=0.20, dividend_yield=float(q))
    call_by_q.append(black_scholes_price(european_call, m))
    put_by_q.append(black_scholes_price(european_put, m))

plt.figure(figsize=(8, 4))
plt.plot(q_grid, call_by_q, marker="o", label="Call europeen")
plt.plot(q_grid, put_by_q, marker="o", label="Put europeen")
plt.xlabel("Dividend yield q")
plt.ylabel("Prix")
plt.title("Impact du dividend yield")
plt.legend()
plt.show()

## 8. Convergence LSMC avec le nombre de chemins

On compare LSMC au prix binomial d'un put americain. Les resultats Monte Carlo restent aleatoires, mais l'erreur standard doit diminuer quand le nombre de chemins augmente.

In [ ]:
reference_tree = binomial_price(american_put, market, steps=750).price
paths_grid = np.array([1_000, 2_500, 5_000, 10_000, 20_000, 50_000])

lsmc_prices = []
lsmc_errors = []

for paths in paths_grid:
    result = lsmc_price(american_put, market, paths=int(paths), steps=75, seed=42)
    lsmc_prices.append(result.price)
    lsmc_errors.append(result.standard_error)

lsmc_prices = np.array(lsmc_prices)
lsmc_errors = np.array(lsmc_errors)

plt.figure(figsize=(9, 4))
plt.errorbar(paths_grid, lsmc_prices, yerr=1.96 * lsmc_errors, marker="o", capsize=4, label="LSMC +/- 1.96 SE")
plt.axhline(reference_tree, color="black", linestyle="--", label="Binomial reference")
plt.xscale("log")
plt.xlabel("Nombre de chemins")
plt.ylabel("Prix")
plt.title("Convergence LSMC")
plt.legend()
plt.show()

list(zip(paths_grid, np.round(lsmc_prices, 6), np.round(lsmc_errors, 6)))

## 9. Dividendes cash discrets

Le binomial et LSMC supportent aussi une liste de dividendes cash discrets `(temps, montant)`.

In [ ]:
cash_div_market = MarketInputs(
    rate=0.05,
    volatility=0.20,
    dividend_yield=0.0,
    discrete_dividends=[(0.25, 1.0), (0.50, 1.0), (0.75, 1.0)],
)

cash_div_american_call = OptionContract(100, 100, 1.0, OptionType.CALL, ExerciseStyle.AMERICAN)
cash_div_american_put = OptionContract(100, 100, 1.0, OptionType.PUT, ExerciseStyle.AMERICAN)

call_cash_div = binomial_price(cash_div_american_call, cash_div_market, steps=750).price
put_cash_div_tree = binomial_price(cash_div_american_put, cash_div_market, steps=750).price
put_cash_div_lsmc = lsmc_price(cash_div_american_put, cash_div_market, paths=30_000, steps=75, seed=123).price

print(f"Call americain avec dividendes cash - binomial : {call_cash_div:.6f}")
print(f"Put americain avec dividendes cash - binomial  : {put_cash_div_tree:.6f}")
print(f"Put americain avec dividendes cash - LSMC      : {put_cash_div_lsmc:.6f}")

assert call_cash_div > 0
assert put_cash_div_tree > 0
assert put_cash_div_lsmc > 0

## 10. Mini regression suite

Cette cellule regroupe quelques checks rapides. Si elle passe, les invariants principaux sont respectes.

In [ ]:
def approx_equal(a, b, tol):
    return abs(a - b) <= tol

checks = []

known_market = MarketInputs(rate=0.05, volatility=0.20)
known_call = OptionContract(100, 100, 1.0, OptionType.CALL, ExerciseStyle.EUROPEAN)
checks.append(("BS known call", approx_equal(black_scholes_price(known_call, known_market), 10.4506, 1e-4)))

checks.append(("Binomial call convergence", approx_equal(binomial_price(european_call, market, 1000).price, bs_call, 0.02)))
checks.append(("American put >= European put", american_put_tree >= tree_put))
checks.append(("American no-div call ~= European call", approx_equal(tree_amer_no_div_call, bs_no_div_call, 0.02)))
checks.append(("Put-call parity", approx_equal(lhs, rhs, 1e-10)))

for name, ok in checks:
    print(f"{name:35s}: {'OK' if ok else 'FAIL'}")

assert all(ok for _, ok in checks)